# Global sizing

The Halo-class unmanned series-hybrid tiltrotor sized in one AeroSandbox/CasADi problem: requirements,
mission, mass closure and energy allocation solved together. Section 1-3 run the current reference (every model on);
sections 4-6 are the framework's sizing building blocks on small example aircraft.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
for path in (repo_root / "src", repo_root):     # this checkout's package even if another one is installed
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
warnings.filterwarnings("ignore", category=FutureWarning)

import aerosandbox.numpy as np
import aerosandbox.tools.units as u
import matplotlib.pyplot as plt

check_log = []


def check(name, passed):
    """Record and print one named verification."""
    passed = bool(passed)
    check_log.append((name, passed))
    print(("PASS  " if passed else "FAIL  ") + name)


ink, ink_muted, grid_color = "#1a1a19", "#5e5d58", "#e4e3dc"
blue, orange, aqua, yellow = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": grid_color, "grid.linewidth": 0.8})

## 1. The reference aircraft

In [ ]:
# The current reference: every model at its highest fidelity (about 10 minutes from cold).
from examples.halo_sizing import (HaloAssumptions as RefAssumptions, HaloRequirements as RefRequirements,
                                  soc_emergency_floor, soc_minimum, solve_halo_sizing as solve_reference)
t0 = time.time()
ref = solve_reference()
print(f"reference: {ref.mass_takeoff_kg:,.0f} kg = {ref.mass_takeoff_kg / u.lbm:,.0f} lb, "
      f"empty {ref.mass_empty_kg:,.0f} kg, fuel {ref.mass_fuel_kg:,.0f} kg ({(time.time() - t0) / 60:.1f} min)")
check("reference closes with every margin satisfied", ref.min_margin > -1e-6)
check("reference take-off weight 15,179 lb (+- 20)", abs(ref.mass_takeoff_kg / u.lbm - 15179) < 20)

In [ ]:
from aerosandbox.library.power_turboshaft import power_turboshaft
r, d, a = ref, ref.design, RefAssumptions()
print(f"wing {d.area_wing_m2:.1f} m2, span {r.span_m:.2f} m, W/S {r.wing_loading_kg_m2:.0f} kg/m2; rotor diameter "
      f"{2 * r.radius_rotor_m:.2f} m, disk loading {r.disk_loading_kg_m2:.0f} kg/m2")
print(f"battery {r.energy_battery_kWh:.1f} kWh; turboshafts 2 x {d.power_rated_turboshaft_W / 1e3:,.0f} kW; "
      f"cruise {r.velocity_cruise_m_s / u.knot:.0f} kt, L/D {r.lift_to_drag_cruise:.2f}")
print("binding:", ", ".join(r.binding))
check("mass closure residual below 1e-5 kg", abs(r.closure_residual_kg) < 1e-5)
check("rotor fits between fuselage and wing tip",
      r.radius_rotor_m <= (r.span_m - a.diameter_fuselage_m) / 2 - a.clearance_rotor_fuselage_m + 1e-6)
check("landing SOC at the 0.30 floor or above", r.soc_end >= soc_minimum - 1e-6)
check("engine-out reserve leaves SOC >= 0.10", r.soc_after_engine_out >= soc_emergency_floor - 1e-6)
check("payload is the 900 kg requirement", abs(r.mass_payload_kg - 900.0) < 1e-6)

**Reading the binding list.** The engine-out hover sizes the battery (its end voltage) and the generators; the
bus-out hover sizes the motors; hover at 4,000 ft sizes the rotors and drive, with the rotor radius capped by the
span; the hot-day hover sizes the heat exchanger; static margin and directional stability size the tails.

## 2. Where the empty mass goes

In [ ]:
c, p = dict(r.component_masses_kg), dict(r.powertrain_masses_kg)
groups = [("Rotors and gearboxes", p["propulsor"] + p["gearbox"] + p["generator_gearbox"], "#1d4f91"),
          ("Motors and generators", p["motor"] + p["generator"], "#2a78d6"),
          ("Battery", p["battery"], "#6fa3e6"),
          ("Turboshafts, cooling,\nprotection", p["turboshaft"] + p["heat_exchanger"] + p["protection_string"]
           + p["protection_bus_tie"] + p["cable_bus_tie"], "#b3cdf0"),
          ("Systems and equipment", c["systems"] + c["equipment"], "#f2b28f"),
          ("Wing, nacelles, tails", c["wing"] + c["nacelles"] + c["horizontal_tail"] + c["vertical_tail"], "#eb6834"),
          ("Fuselage", c["fuselage"], "#b84a1f"),
          ("Landing gear", c["landing_gear"], "#7a3314")]
oew = sum(kg for _, kg, _ in groups)
check("empty groups sum to the empty mass", abs(oew - r.mass_empty_kg) < 1e-3)
fig, ax = plt.subplots(figsize=(8.5, 5.8))
wedges, _ = ax.pie([kg for _, kg, _ in groups], colors=[col for *_, col in groups], startangle=90,
                   counterclock=False, wedgeprops=dict(width=0.42, edgecolor="white", linewidth=2))
for wedge, (label, kg, _) in zip(wedges, groups):
    angle = np.deg2rad((wedge.theta1 + wedge.theta2) / 2)
    x, y = np.cos(angle), np.sin(angle)
    ax.annotate(f"{label}\n{kg:,.0f} kg ({100 * kg / oew:.0f} %)", xy=(0.95 * x, 0.95 * y),
                xytext=(1.3 * x, 1.25 * y), ha="left" if x > 0 else "right", va="center", fontsize=9,
                arrowprops=dict(arrowstyle="-", color=ink_muted, lw=0.8))
ax.text(0, 0.08, f"{oew:,.0f} kg", ha="center", va="center", fontsize=15, fontweight="bold")
ax.text(0, -0.12, "empty mass", ha="center", va="center", fontsize=9, color=ink_muted)
ax.set(aspect="equal")
ax.set_title("Empty mass (powertrain blue, airframe orange)", pad=28)
plt.show()
print(f"powertrain share of empty mass: {sum(kg for _, kg, _ in groups[:4]) / oew:.0%}")

## 3. Mission and energy allocation

In [ ]:
for s in r.segments:
    print(f"  {s['label']:<15}{s['duration_s'] / 60:6.1f} min {s['fuel_kg']:7.1f} kg fuel  SOC {s['soc_end']:.3f}  "
          f"battery share {s['hybridization']:.2f}  {s['power_rotors_W'] / 1e3:6.0f} kW")
# A negative battery share is the generators recharging the battery in flight.
check("battery never supplies more than the rotors need (share <= 1)",
      all(s["hybridization"] <= 1 + 1e-6 for s in r.segments))
check("fuel load is 1.1 x fuel burnt", abs(r.mass_fuel_kg - 1.1 * r.mass_fuel_burnt_kg) < 1e-3)
t = np.cumsum([0] + [s["duration_s"] / 60 for s in r.segments])
fig, axes = plt.subplots(2, 1, figsize=(8, 5.2), sharex=True)
axes[0].step(t[:-1], [s["power_rotors_W"] / 1e3 for s in r.segments], where="post", color=orange, lw=2)
axes[0].set(ylabel="rotor power [kW]", title="The 445 nm mission")
axes[1].plot(t, [0.95] + [s["soc_end"] for s in r.segments], color=blue, lw=2, marker="o", ms=6, mec="white")
axes[1].axhline(soc_minimum, color=ink_muted, lw=1, ls="--")
axes[1].set(xlabel="mission time [min]", ylabel="battery SOC", ylim=(0, 1))
plt.tight_layout()
plt.show()

## 4. Requirements: flight points and powertrain sizing

*From `tier7_requirements/requirements_verification.ipynb`.*

In [ ]:
import ast
import sys
from dataclasses import replace
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import aerosandbox as asb
import aerosandbox.numpy as np
import matplotlib.pyplot as plt

from aircraft_closure.aerodynamics.simple import SimpleAerodynamics
from aircraft_closure.core.margins import margin_report
from aircraft_closure.performance.flight_point import FlightCondition, acceleration_gravity_m_s2, build_flight_point
from aircraft_closure.requirements.capability import RequirementSet, SpeedRequirement
from examples.aircraft_mass_closure import build_reference_aircraft
from examples.requirements_sizing import solve_requirements_sizing

check_log = globals().get("check_log", [])


def check(name, passed):
    """Record and print one named verification."""
    passed = bool(passed)
    check_log.append((name, passed))
    print(("PASS  " if passed else "FAIL  ") + name)


def close(a, b, rel=1e-9, abs_tol=1e-12):
    return abs(a - b) <= max(rel * abs(b), abs_tol)


aircraft = build_reference_aircraft(3.23, area_horizontal_tail_m2=2.16, area_vertical_tail_m2=1.92)
aero = SimpleAerodynamics()
mass_kg = 1553.0
weight_N = mass_kg * acceleration_gravity_m_s2


def solve_point(condition, hybridization=None, minimize_fuel=True):
    opti = asb.Opti()
    point = build_flight_point(opti, aircraft, aero, condition, mass_kg, hybridization)
    if minimize_fuel:
        opti.minimize(point.fuel_flow_kg_s * 100)
    return point, opti.solve(verbose=False)


ink, ink_muted, grid_color = "#1a1a19", "#5e5d58", "#e4e3dc"
blue, orange, aqua = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": grid_color, "grid.linewidth": 0.8})

### 1. Flight-point identities

Hover thrust sums to $(T/W)W$ and its shaft power equals the closed-form
actuator-disk value; airplane-mode lift and thrust satisfy the flight-path force
balance; the bus splits motor demand by $h_e$; fuel flow is the turboshaft model
at the generator shaft power.

In [ ]:
hover, s = solve_point(FlightCondition(mode="hover", altitude_m=0.0, label="hover"))
rotor_model = aircraft.powertrain.topology.instances["propulsor"].component
expected_W = float(rotor_model.evaluate(0, asb.Atmosphere(altitude=0), thrust_N=weight_N / 4).shaft_power_W)
check("hover: thrust per rotor x 4 = weight", close(float(s.value(4 * hover.thrust_per_rotor_N)), weight_N, rel=1e-9))
check("hover: rotor shaft power = actuator-disk closed form", close(float(s.value(hover.power_shaft_rotor_W)), expected_W, rel=1e-9))
check("hover: motor shaft power = rotor power / gearbox efficiency",
      close(float(s.value(hover.speed_motor_rad_s * hover.torque_motor_Nm)), expected_W / 0.97, rel=1e-9))

climb, s = solve_point(FlightCondition(velocity_m_s=50.0, climb_rate_m_s=5.0, label="climb"), hybridization=0.0)
sin_gamma = 0.1
check("airplane: lift = W cos(gamma)", close(float(s.value(climb.aero.lift_N)), weight_N * np.sqrt(1 - sin_gamma**2), rel=1e-9))
check("airplane: thrust = D + W sin(gamma)",
      close(float(s.value(4 * climb.thrust_per_rotor_N)), float(s.value(climb.aero.drag_N)) + weight_N * sin_gamma, rel=1e-9))
check("airplane: h_e = 0 draws no battery power", abs(float(s.value(climb.power_battery_W))) < 1e-6)

cruise, s = solve_point(FlightCondition(velocity_m_s=60.0, label="cruise"), hybridization=0.4)
demand_W = float(s.value(cruise.power_electric_motors_W))
check("bus: battery supplies h_e of motor demand", close(float(s.value(cruise.power_battery_W)), 0.4 * demand_W, rel=1e-8))
check("bus: generator supplies the rest", close(float(s.value(cruise.generator.power_electric_W)), 0.6 * demand_W, rel=1e-8))
check("margins: every margin is labelled with its point", all(m.label.startswith("cruise: ") for m in cruise.margins))

### 2. Power required across the envelope

Hover power rises with density altitude; airplane-mode shaft power follows the
drag-power curve plus the climb term $W\dot h$; the electrical input adds motor and
gearbox losses. These are the curves the requirements sample.

In [ ]:
altitudes = np.linspace(0, 4000, 9)
hover_kW = []
for h in altitudes:
    p, s = solve_point(FlightCondition(mode="hover", altitude_m=h, label="hover"))
    hover_kW.append(4 * float(s.value(p.power_shaft_rotor_W)) / 1e3)
check("trend: hover power rises with altitude", np.all(np.diff(hover_kW) > 0))
speeds = np.linspace(42, 85, 12)
curves = {}
for rate, color in ((0.0, blue), (5.0, orange)):
    shaft = []
    for v in speeds:
        p, s = solve_point(FlightCondition(velocity_m_s=v, altitude_m=1000.0, climb_rate_m_s=rate, label="pt"), hybridization=0.0)
        shaft.append(4 * float(s.value(p.power_shaft_rotor_W)) / 1e3)
    curves[rate] = np.array(shaft)
check("trend: 5 m/s climb adds about W x climb rate / rotor efficiency",
      np.all(curves[5.0] - curves[0.0] > weight_N * 5.0 / 1e3 * 0.95))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(altitudes, hover_kW, color=blue, lw=2)
ax1.set(xlabel="altitude [m]", ylabel="total rotor shaft power [kW]", title=f"Hover at {mass_kg:.0f} kg")
for rate, color in ((0.0, blue), (5.0, orange)):
    ax2.plot(speeds, curves[rate], color=color, lw=2, label=f"climb {rate:.0f} m/s")
ax2.set(xlabel="airspeed [m/s]", ylabel="total rotor shaft power [kW]", title="Airplane mode, 1000 m")
ax2.legend(frameon=False)
plt.show()

### 3. Requirements on the fixed Tier 6 aircraft

With the reference ratings (100 kW motors, 400 kW generator, 600 kW turboshaft),
each requirement point is solved with its margins as constraints; feasibility
and the smallest margin per point are reported.

In [ ]:
requirements = RequirementSet()
for condition in requirements.flight_conditions():
    opti = asb.Opti()
    point = build_flight_point(opti, aircraft, aero, condition, mass_kg)
    opti.subject_to([m.value >= 0 for m in point.margins])
    opti.minimize(point.fuel_flow_kg_s * 100)
    s = opti.solve(verbose=False)
    worst = margin_report(point.margins, s.value)[0]
    print(f"{condition.label:<10} shaft/rotor {float(s.value(point.power_shaft_rotor_W)) / 1e3:6.1f} kW, "
          f"h_e {float(s.value(point.hybridization_electric)):.2f}, smallest margin {worst.label} {float(worst.value):+.3f}")
    check(f"fixed aircraft: {condition.label} requirement feasible", float(worst.value) > -1e-6)

### 4. Reference case: sizing the powertrain to the requirements

Minimum MTOM over wing position, motor and generator peak torques, turboshaft
rating, battery discharge power and rotor disk area (bounded at 15 m2), with mass
closure, hover pitch trim and every requirement margin $\ge 0$. Max speed on
turbogenerator power sizes the turboshaft and generator; hover sizes the motor,
gearbox and rotor ratings. Point requirements carry no energy cost, so the
optimizer accepts a high disk loading; Tier 8/9 adds hover energy.

In [ ]:
sized = solve_requirements_sizing()
binding = sorted(label for label, m in sized.margins if abs(m) < 1e-6)
print("binding:", binding)
print(f"MTOM {sized.mass_takeoff_kg:.1f} kg; motor {sized.power_rated_motor_W / 1e3:.1f} kW, generator "
      f"{sized.power_rated_generator_W / 1e3:.1f} kW, turboshaft {sized.power_rated_turboshaft_W / 1e3:.1f} kW, "
      f"battery {sized.power_max_discharge_battery_W / 1e3:.1f} kW, disk {sized.area_disk_m2:.2f} m2, hover h_e {sized.hybridization_hover:.2f}")
check("sizing: every requirement margin non-negative", all(m > -1e-6 for _, m in sized.margins))
check("sizing: max speed sizes the turboshaft", "max_speed: turboshaft power_shaft_W" in binding)
check("sizing: hover sizes the motor", "hover: motor power_shaft_W" in binding)
check("sizing: sustained points draw no battery power",
      all(abs(b) < 1e-6 for label, _, b, _ in sized.point_powers_W if label != "hover"))
check("sizing: turboshaft rated to the generator", close(sized.power_rated_turboshaft_W, sized.power_rated_generator_W, rel=1e-6))
check("sizing: masses sum to MTOM", close(sum(m for _, m in sized.component_masses_kg), sized.mass_takeoff_kg, rel=1e-9))
check("sizing: resized powertrain is lighter than the fixed reference (880 kg)",
      dict(sized.component_masses_kg)["powertrain"] < 880.0)

labels = ["motor (each)", "generator", "turboshaft", "battery discharge"]
reference_kW = [100, 400, 600, 400]
sized_kW = [sized.power_rated_motor_W / 1e3, sized.power_rated_generator_W / 1e3,
            sized.power_rated_turboshaft_W / 1e3, sized.power_max_discharge_battery_W / 1e3]
y = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.barh(y + 0.18, reference_kW, height=0.34, color=ink_muted, label="Tier 4 reference")
ax.barh(y - 0.18, sized_kW, height=0.34, color=blue, label="sized to requirements")
ax.set_yticks(y, labels)
ax.set(xlabel="rating [kW]", title="Powertrain ratings")
ax.grid(axis="y", visible=False)
ax.legend(frameon=False, loc="lower right")
plt.show()

In [ ]:
# Requirement sweep: the speed requirement sets the turbogenerator rating.
speeds_req = [70.0, 80.0, 90.0]
sweep = [solve_requirements_sizing(replace(RequirementSet(), speed=SpeedRequirement(velocity_m_s=v))) for v in speeds_req]
for v, r in zip(speeds_req, sweep):
    print(f"V_max {v:.0f} m/s: turboshaft {r.power_rated_turboshaft_W / 1e3:6.1f} kW, MTOM {r.mass_takeoff_kg:7.1f} kg")
check("sweep: turboshaft rating rises with required speed", np.all(np.diff([r.power_rated_turboshaft_W for r in sweep]) > 0))
check("sweep: MTOM rises with required speed", np.all(np.diff([r.mass_takeoff_kg for r in sweep]) > 0))

## 5. Missions

*From `tier8_missions/mission_verification.ipynb`.*

In [ ]:
import ast
import sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import aerosandbox as asb
import aerosandbox.numpy as np
import matplotlib.pyplot as plt

from aircraft_closure.aerodynamics.simple import SimpleAerodynamics
from aircraft_closure.mission.mission import Mission, build_mission
from aircraft_closure.mission.segments import (ClimbSegment, CruiseSegment, DescentSegment, HoverSegment,
                                               LoiterSegment, ground_distance_m)
from examples.aircraft_mass_closure import build_reference_aircraft
from examples.mission_analysis import (fuel_factor, reference_mission, soc_minimum, solve_mission,
                                       solve_prescribed_mission, solve_semi_free_mission)

check_log = globals().get("check_log", [])


def check(name, passed):
    """Record and print one named verification."""
    passed = bool(passed)
    check_log.append((name, passed))
    print(("PASS  " if passed else "FAIL  ") + name)


def close(a, b, rel=1e-9, abs_tol=1e-12):
    return abs(a - b) <= max(rel * abs(b), abs_tol)


aircraft = build_reference_aircraft(3.23, area_horizontal_tail_m2=2.16, area_vertical_tail_m2=1.92)
aero = SimpleAerodynamics()
battery = aircraft.powertrain.topology.instances["battery"].component
ink, ink_muted, grid_color = "#1a1a19", "#5e5d58", "#e4e3dc"
blue, orange, aqua = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": grid_color, "grid.linewidth": 0.8})


def fly(segments, mass_kg=1550.0, soc=0.9):
    opti = asb.Opti()
    flown = build_mission(opti, aircraft, aero, Mission(tuple(segments)), mass_kg, soc)
    opti.minimize(flown.mass_fuel_burnt_kg)
    return flown, opti.solve(verbose=False)

### 1. M1: isolated segments

Each segment is verified on its own: duration, ground distance, the fuel
identity (flow x time), the battery identity (SOC drop = chemical energy /
capacity, including Joule losses) and the hover/airplane mapping.

In [ ]:
check("segment: climb duration = delta h / rate", ClimbSegment(0, 1000, 4.0).duration_s() == 250.0)
check("segment: cruise duration = distance / V", CruiseSegment(90000.0, velocity_m_s=60.0).duration_s() == 1500.0)
check("segment: descent evaluated at mean altitude with negative climb rate",
      DescentSegment(1000, 200, 2.0).flight_condition(0.8).altitude_m == 600
      and DescentSegment(1000, 200, 2.0).flight_condition(0.8).climb_rate_m_s == -2.0)
check("segment: climb ground distance uses the horizontal speed",
      close(float(ground_distance_m(ClimbSegment(0, 1000, 3.0, 50.0))), np.sqrt(50.0**2 - 9) * 1000 / 3))

flown, s = fly([CruiseSegment(60000.0, 1000.0, 60.0, 0.0)])
seg = flown.segments[0]
check("cruise: fuel = fuel flow x distance / V", close(float(s.value(seg.mass_fuel_burnt_kg)),
                                                       float(s.value(seg.point.fuel_flow_kg_s)) * 1000.0))
check("cruise: turbo-only segment leaves SOC unchanged", close(float(s.value(seg.soc_end)), 0.9))
flown, s = fly([HoverSegment(60.0, 0.0, 1.0)])
seg = flown.segments[0]
check("hover: SOC drop = chemical energy / capacity",
      close(0.9 - float(s.value(seg.soc_end)), float(s.value(seg.energy_battery_chemical_J)) / battery.energy_capacity_J))
check("hover: chemical energy exceeds terminal energy (Joule losses)",
      float(s.value(seg.energy_battery_chemical_J)) > float(s.value(seg.energy_battery_terminal_J)))
check("hover: all-electric hover burns only generator no-load fuel",
      float(s.value(seg.mass_fuel_burnt_kg)) < 0.05)

### 2. M2: prescribed mission

Hover at 70 % battery power; every other segment on the turbogenerator; all
speeds fixed. Take-off mass, fuel load and wing position close with the mission
in one solve.

In [ ]:
prescribed = solve_prescribed_mission()
print(f"MTOM {prescribed.mass_takeoff_kg:.1f} kg, fuel load {prescribed.mass_fuel_kg:.2f} kg "
      f"(burnt {prescribed.mass_fuel_burnt_kg:.2f} kg), SOC at landing {prescribed.soc_end:.3f}, "
      f"time {prescribed.duration_s / 60:.1f} min")
for label, duration, fuel, energy, soc, h_e, shaft in prescribed.segments:
    print(f"  {label:<15}{duration:7.0f} s {fuel:7.2f} kg {energy / 3.6e6:6.2f} kWh  SOC {soc:.3f}  h_e {h_e:.2f}  "
          f"{shaft / 1e3:5.1f} kW/rotor")
check("M2: fuel load = 1.1 x burnt fuel", close(prescribed.mass_fuel_kg, fuel_factor * prescribed.mass_fuel_burnt_kg, rel=1e-8))
check("M2: battery used only in hover", all(abs(e) < 1e-3 for l, _, _, e, _, _, _ in prescribed.segments if "hover" not in l))
check("M2: every segment margin non-negative", prescribed.min_margin > -1e-6)
check("M2: total time is the sum of segment durations",
      close(prescribed.duration_s, 60 + 250 + 100000 / 60 + 1200 + 1000 / 3 + 60))
check("M2: cruise burns the most fuel", max(prescribed.segments, key=lambda r: r[2])[0] == "cruise")

### 3. M3: semi-free mission (minimum fuel)

Cruise speed (40-85 m/s) and every segment's electric power fraction are free.
The optimizer slows cruise toward the minimum-fuel speed and spends the battery
down to its SOC floor. Turboshaft efficiency is constant at this tier, so the
fuel saving comes from replacing turbogenerator energy with stored energy, and
from speed.

In [ ]:
free = solve_semi_free_mission()
print(f"MTOM {free.mass_takeoff_kg:.1f} kg, fuel load {free.mass_fuel_kg:.2f} kg, SOC at landing {free.soc_end:.3f}, "
      f"cruise {free.velocity_cruise_m_s:.1f} m/s")
check("M3: less fuel than the prescribed mission", free.mass_fuel_kg < prescribed.mass_fuel_kg)
check("M3: battery spent to the SOC floor", close(free.soc_end, soc_minimum, rel=1e-6))
check("M3: lighter take-off (less fuel carried)", free.mass_takeoff_kg < prescribed.mass_takeoff_kg)
check("M3: margins non-negative", free.min_margin > -1e-6)

def timeline(result):
    t, soc, mass = [0.0], [0.95], [result.mass_takeoff_kg]
    for label, duration, fuel, energy, soc_end, _, _ in result.segments:
        t.append(t[-1] + duration)
        soc.append(soc_end)
        mass.append(mass[-1] - fuel)
    return np.array(t) / 60, np.array(soc), np.array(mass)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
for result, label, color in ((prescribed, "prescribed (M2)", blue), (free, "semi-free (M3)", orange)):
    t, soc, mass = timeline(result)
    ax1.plot(t, soc, color=color, lw=2, marker="o", ms=6, mec="white", mew=1.5, label=label)
    ax2.plot(t, mass, color=color, lw=2, marker="o", ms=6, mec="white", mew=1.5, label=label)
ax1.axhline(soc_minimum, color=ink_muted, lw=1, ls="--")
ax1.text(1, soc_minimum + 0.01, "SOC floor", color=ink_muted, fontsize=9)
ax1.set(xlabel="mission time [min]", ylabel="battery SOC", title="State of charge")
ax2.set(xlabel="mission time [min]", ylabel="aircraft mass [kg]", title="Mass (fuel burn)")
ax1.legend(frameon=False, loc="center left")
plt.show()

In [ ]:
# Fuel per 100 km against a prescribed cruise speed (turbo-only cruise, closed mass).
speeds = [45.0, 50.0, 55.0, 60.0, 70.0, 80.0]
fuel_kg = [solve_mission(reference_mission(velocity_cruise_m_s=v)).mass_fuel_burnt_kg for v in speeds]
best = speeds[int(np.argmin(fuel_kg))]
print("mission fuel by cruise speed:", dict(zip(speeds, np.round(fuel_kg, 2))))
check("speed sweep: an interior minimum-fuel cruise speed exists", speeds[0] < best < speeds[-1])
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(speeds, fuel_kg, color=blue, lw=2, marker="o", ms=8, mec="white", mew=2)
ax.set(xlabel="cruise speed [m/s]", ylabel="mission fuel burnt [kg]", title="Prescribed mission: fuel against cruise speed")
plt.show()

## 6. Coupled sizing and energy allocation

*From `tier9_coupled_sizing/coupled_sizing_verification.ipynb`.*

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import aerosandbox.numpy as np
import matplotlib.pyplot as plt

from aircraft_closure.mission.mission import Mission
from aircraft_closure.mission.segments import CruiseSegment
from aircraft_closure.powertrain.components.battery import Battery
from examples.coupled_sizing import soc_emergency_floor, solve_coupled_sizing
from examples.mission_analysis import fuel_factor, reference_mission, soc_minimum

check_log = globals().get("check_log", [])


def check(name, passed):
    """Record and print one named verification."""
    passed = bool(passed)
    check_log.append((name, passed))
    print(("PASS  " if passed else "FAIL  ") + name)


def close(a, b, rel=1e-9, abs_tol=1e-12):
    return abs(a - b) <= max(rel * abs(b), abs_tol)


ink, ink_muted, grid_color = "#1a1a19", "#5e5d58", "#e4e3dc"
blue, orange, aqua = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": grid_color, "grid.linewidth": 0.8})

### 1. Smooth battery mass

`Battery.get_mass` keeps the exact maximum of energy- and power-sized mass by
default (Tier 1 identities unchanged). With `mass_smoothing_kg` set, it uses
AeroSandbox's softmax, which is never below the exact value and at most
smoothing x ln 2 above it. Without it, the coupled optimum sits on the kink and
IPOPT stalls.

In [ ]:
powers_W = np.linspace(2e4, 3e5, 60)
exact = np.array([Battery(max_discharge_power_W=p).get_mass() for p in powers_W])
smooth = np.array([float(Battery(max_discharge_power_W=p, mass_smoothing_kg=2.0).get_mass()) for p in powers_W])
check("battery: smooth mass never below the exact maximum", np.all(smooth >= exact - 1e-12))
check("battery: smooth mass within 2 kg x ln 2 of the exact maximum", np.all(smooth - exact <= 2 * np.log(2) + 1e-12))
check("battery: default stays the exact maximum", Battery().get_mass() == max(36e6 / 9e5, 1e5 / 3000))

### 2. Reference case: minimum take-off mass

The optimizer resizes everything at once. The margin report names what binds:
the max-speed requirement sizes the turbogenerator, hover sizes motors, gearboxes
and rotors, the engine-out reserve sizes the battery (both power and energy),
static margin and $C_{n\beta}$ size the tails, and the mission's SOC floor binds
because the optimizer spends the battery it must carry anyway.

In [ ]:
light = solve_coupled_sizing("mass_takeoff")
print(f"MTOM {light.mass_takeoff_kg:.1f} kg, empty {light.mass_empty_kg:.1f} kg, fuel {light.mass_fuel_kg:.2f} kg, "
      f"battery {light.energy_battery_kWh:.1f} kWh / {light.power_max_discharge_battery_W / 1e3:.0f} kW")
print(f"wing {light.area_wing_m2:.2f} m2, tails {light.area_horizontal_tail_m2:.2f} / {light.area_vertical_tail_m2:.2f} m2, "
      f"disk {light.area_disk_m2:.2f} m2 per rotor")
print(f"motor {light.power_rated_motor_W / 1e3:.1f} kW, generator {light.power_rated_generator_W / 1e3:.1f} kW, "
      f"turboshaft {light.power_rated_turboshaft_W / 1e3:.1f} kW; cruise {light.velocity_cruise_m_s:.1f} m/s, L/D {light.lift_to_drag_cruise:.2f}")
print("binding:", ", ".join(light.binding))
check("closure: residual below 1e-5 kg", abs(light.closure_residual_kg) < 1e-5)
check("closure: masses sum to MTOM", close(sum(m for _, m in light.component_masses_kg), light.mass_takeoff_kg, rel=1e-8))
check("constraints: every margin non-negative", light.min_margin > -1e-6)
check("constraints: fuel load = 1.1 x mission fuel", close(light.mass_fuel_kg, fuel_factor * light.mass_fuel_burnt_kg, rel=1e-7))
check("constraints: landing SOC at the floor", close(light.soc_end, soc_minimum, rel=1e-6))
check("constraints: static margin and Cn_beta met", light.static_margin >= 0.1 - 1e-7 and light.cn_beta_per_rad >= 0.06 - 1e-7)
check("constraints: failed-rotor rudder within 20 deg", light.rudder_failed_rotor_deg <= 20 + 1e-6)
for label in ("max_speed: turboshaft power_shaft_W", "hover: motor power_shaft_W", "soc_after_engine_out_hover",
              "engine-out hover: battery discharge_power_W", "static_margin", "cn_beta_per_rad", "soc_end"):
    check(f"binding: {label}", label in light.binding)

In [ ]:
items = sorted(((n, m) for n, m in light.component_masses_kg if m > 0), key=lambda x: x[1])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))
ax1.barh([n for n, _ in items], [m for _, m in items], color=blue, height=0.6)
for y, (_, m) in enumerate(items):
    ax1.text(m + 5, y, f"{m:.0f}", va="center", fontsize=9, color=ink)
ax1.set(xlabel="mass [kg]", xlim=(0, max(m for _, m in items) * 1.15), title=f"Min-MTOM breakdown ({light.mass_takeoff_kg:.0f} kg)")
ax1.grid(axis="y", visible=False)
labels = [s[0] for s in light.segments]
ax2.bar(np.arange(len(labels)), [s[5] for s in light.segments], color=orange, width=0.6)
ax2.set_xticks(np.arange(len(labels)), labels, rotation=30, ha="right")
ax2.set(ylabel="electric power fraction h_e", ylim=(0, 1), title="Energy allocation per segment (min MTOM)")
plt.show()
check("allocation: every h_e within [0, 1]", all(-1e-9 <= s[5] <= 1 + 1e-9 for s in light.segments))
check("allocation: the carried battery is used during the mission", sum(s[3] for s in light.segments) > 0)

### 3. Trade case: minimum fuel

Same constraints, objective mission fuel. The optimizer flies almost entirely on
the battery, which then sizes far larger and drags MTOM, wing and rotors up
with it: the classic mass-versus-energy trade of a hybrid, now produced by one
problem rather than by an outer sweep.

In [ ]:
frugal = solve_coupled_sizing("fuel")
rows = [("MTOM [kg]", light.mass_takeoff_kg, frugal.mass_takeoff_kg),
        ("fuel load [kg]", light.mass_fuel_kg, frugal.mass_fuel_kg),
        ("battery [kWh]", light.energy_battery_kWh, frugal.energy_battery_kWh),
        ("wing [m2]", light.area_wing_m2, frugal.area_wing_m2),
        ("disk per rotor [m2]", light.area_disk_m2, frugal.area_disk_m2),
        ("turboshaft [kW]", light.power_rated_turboshaft_W / 1e3, frugal.power_rated_turboshaft_W / 1e3),
        ("cruise speed [m/s]", light.velocity_cruise_m_s, frugal.velocity_cruise_m_s),
        ("cruise L/D", light.lift_to_drag_cruise, frugal.lift_to_drag_cruise)]
print(f"{'':<22}{'min MTOM':>12}{'min fuel':>12}")
for name, a, b in rows:
    print(f"{name:<22}{a:12.2f}{b:12.2f}")
check("trade: min-fuel design burns less fuel", frugal.mass_fuel_kg < light.mass_fuel_kg)
check("trade: min-fuel design carries a larger battery", frugal.energy_battery_kWh > light.energy_battery_kWh)
check("trade: min-fuel design is heavier", frugal.mass_takeoff_kg > light.mass_takeoff_kg)
check("trade: min-fuel constraints all satisfied", frugal.min_margin > -1e-6 and abs(frugal.closure_residual_kg) < 1e-5)

fig, ax = plt.subplots(figsize=(7.5, 3.8))
for result, label, color in ((light, "min MTOM", blue), (frugal, "min fuel", orange)):
    t, soc = [0.0], [0.95]
    for s in result.segments:
        t.append(t[-1] + s[1] / 60)
        soc.append(s[4])
    ax.plot(t, soc, color=color, lw=2, marker="o", ms=6, mec="white", mew=1.5, label=label)
ax.axhline(soc_minimum, color=ink_muted, lw=1, ls="--")
ax.axhline(soc_emergency_floor, color=ink_muted, lw=1, ls=":")
ax.text(1, soc_minimum + 0.01, "mission SOC floor", color=ink_muted, fontsize=9)
ax.text(1, soc_emergency_floor + 0.01, "engine-out reserve floor", color=ink_muted, fontsize=9)
ax.set(xlabel="mission time [min]", ylabel="battery SOC", ylim=(0, 1), title="State of charge along the optimized missions")
ax.legend(frameon=False, loc="center left")
plt.show()

## Summary

In [ ]:
failed = [name for name, passed in check_log if not passed]
print(f"{len(check_log) - len(failed)} of {len(check_log)} checks passed")
for name in failed:
    print("FAILED:", name)